In [0]:
%run ./config

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
def get_watermark(table_name):
    watermark_path = f"{base_path}metadata/watermark"
    watermark_df = (
        spark.read
        .format("delta")
        .options(**storage_options)
        .load(watermark_path)
    )
    row = (
        watermark_df
        .filter(col("table_name") == table_name)
        .select("watermark_value")
        .first()
    )
    if row:
        return row[0]
    return None

def update_watermark(table_name, watermark_value):
    watermark_path = f"{base_path}metadata/watermark"
    watermark_df = (
        spark.read
        .format("delta")
        .options(**storage_options)
        .load(watermark_path)
    )
    watermark_df = watermark_df.filter(
        col("table_name") != table_name
    )
    new_record = spark.createDataFrame(
        [(table_name, watermark_value)],
        ["table_name", "watermark_value"]
    )
    final_df = watermark_df.union(new_record)
    final_df.write.format("delta").mode("overwrite").options(**storage_options).save(watermark_path)

In [0]:
def write_audit_log(table_name,records_read,records_valid,records_rejected,status):
    audit_path = f"{base_path}metadata/audit_log"
    audit_df = spark.createDataFrame(
        [(
            table_name,
            records_read,
            records_valid,
            records_rejected,
            status
        )],
        [
            "table_name",
            "records_read",
            "records_valid",
            "records_rejected",
            "status"
        ]
    )
    audit_df = audit_df.withColumn("load_timestamp",current_timestamp())
    audit_df.write.format("delta").mode("append").options(**storage_options).save(audit_path)

In [0]:
def apply_type_cast(df, table_name):
    cfg = METADATA_CONFIG[table_name]
    for column_info in cfg["columns"]:
        col_name = column_info["name"]
        data_type = column_info["type"].lower()
        if data_type == "date":
            formats = column_info.get("date_formats") or []
            parsed_date = None
            for fmt in formats:
                current_parse = expr(
                    f"try_to_timestamp({col_name}, '{fmt}')"
                ).cast("date")
                if parsed_date is None:
                    parsed_date = current_parse
                else:
                    parsed_date = coalesce(parsed_date,current_parse)
            df = df.withColumn(col_name,parsed_date)
        else:
            df = df.withColumn(col_name,col(col_name).cast(data_type))
    return df

In [0]:
def validate_records(df, table_name):
    cfg = METADATA_CONFIG[table_name]
    pk  = cfg["primary_key"]
    
    def get_rule(column, rule):
        rule_map = {
            "not_null"    : (col(column).isNull(),
                             f"{column} is NULL"),
            "not_empty"   : (col(column).isNull() | (trim(col(column)) == ""),
                             f"{column} is NULL or empty"),
            "positive"    : (col(column).isNull() | (col(column) <= 0),
                             f"{column} must be positive"),
            "not_future"  : (col(column).isNull() | (col(column) > current_date()),
                             f"{column} cannot be a future date"),
            "valid_email" : (~col(column).rlike("^[A-Za-z0-9._%+\\-]+@[A-Za-z0-9.\\-]+\\.[A-Za-z]{2,}$"),
                             f"{column} format is invalid"),
            "valid_phone" : (
                             col(column).isNull()
                             | (trim(col(column)) == "")
                             | (length(regexp_replace(col(column), "[^0-9]", "")) < 7)
                             | (length(regexp_replace(col(column), "[^0-9]", "")) > 15)
                             | (~col(column).rlike("^[0-9+()\\-. xX]+$")),
                             f"{column} format is invalid"
                             ),
        }
        return rule_map.get(rule)

    pk_counts = df.groupBy(pk).agg(count("*").alias("_pk_count"))
    df = df.join(pk_counts,pk,"left")

    pk_null_condition = col(pk).isNull()
    pk_dup_condition = col("_pk_count") > 1
    reject_expr = (
        when(pk_null_condition, lit(f"{pk} is NULL (PK violation)"))
        .when(pk_dup_condition, lit(f"{pk} is duplicated (PK violation)"))
        .otherwise(lit(None).cast("string"))
    )

    validations = cfg.get("validations", [])

    for v in validations:
        column  = v["column"]
        rule    = v["rule"]
        result  = get_rule(column, rule)
        if result is None:
            print(f"[WARN] Unknown rule '{rule}' for column '{column}' — skipping")
            continue
        condition, message = result
        reject_expr = when(
            reject_expr.isNull() & condition,
            lit(message)
        ).otherwise(reject_expr)
    df = df.withColumn("_RejectReason", reject_expr).drop("_pk_count")
    df_reject = df.filter(col("_RejectReason").isNotNull())
    df_valid  = df.filter(col("_RejectReason").isNull()).drop("_RejectReason")

    return df_valid, df_reject

In [0]:
def validate_foreign_keys(df, table_name):
    cfg = METADATA_CONFIG[table_name]
    if len(cfg["foreign_keys"]) == 0:
        return df, spark.createDataFrame([], df.schema)
    valid_df = df
    reject_df = None
    for fk in cfg["foreign_keys"]:
        fk_column = fk["column"]
        parent_table = fk["parent_table"]
        parent_key = fk["parent_key"]
        parent_cfg = METADATA_CONFIG[parent_table]
        parent_path = f"{base_path}{parent_cfg['silver_path']}"
        parent_df = (spark.read.format("delta").options(**storage_options).load(parent_path)
            .select(parent_key)
            .distinct()
        )
        invalid_rows = (
            valid_df.alias("child")
            .join(parent_df.alias("parent"),
                col(f"child.{fk_column}") == col(f"parent.{parent_key}"),
                "leftanti"
            )
            .withColumn("_RejectReason",lit(f"{fk_column} FK Validation Failed"))
        )
        if reject_df is None:
            reject_df = invalid_rows
        else:
            reject_df = reject_df.unionByName(
                invalid_rows,
                allowMissingColumns=True
            )
        valid_df = (
            valid_df.alias("child")
            .join(
                parent_df.alias("parent"),
                col(f"child.{fk_column}") == col(f"parent.{parent_key}"),
                "leftsemi"
            )
        )
    if reject_df is None:
        reject_df = spark.createDataFrame([], valid_df.schema)

    return valid_df, reject_df

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, ArrayType

def flatten_complete(df):
    while True:
        struct_cols = [
            field.name
            for field in df.schema.fields
            if isinstance(field.dataType, StructType)
        ]
        array_cols = [
            field.name
            for field in df.schema.fields
            if isinstance(field.dataType, ArrayType)
        ]
        if not struct_cols and not array_cols:
            break
        for col_name in array_cols:
            df = df.withColumn(
                col_name,
                explode_outer(col(col_name))
            )
        for col_name in struct_cols:
            expanded_cols = [
                col(f"{col_name}.{field.name}")
                .alias(f"{col_name}_{field.name}")
                for field in df.schema[col_name].dataType.fields
            ]
            remaining_cols = [
                c for c in df.columns
                if c != col_name
            ]
            df = df.select(
                *remaining_cols,
                *expanded_cols
            )

    return df

In [0]:
from pyspark.sql.functions import *

def normalize_exchange_rates(df):
    rate_columns = [
        c for c in df.columns
        if c.startswith("rates_")
    ]
    stack_parts = []
    for c in rate_columns:
        currency = c.replace("rates_", "")
        stack_parts.append(
            f"'{currency}', CAST({c} AS DECIMAL(18,6))"
        )
    stack_expr = ",".join(stack_parts)
    df_normalized = (
        df.selectExpr(
            "base_code as BaseCurrency",
            "time_last_update_utc",
            f"""
            stack(
                {len(rate_columns)},
                {stack_expr}
            ) as (TargetCurrency, ExchangeRate)
            """
        )
    )
    df_normalized = (
        df_normalized
        .withColumn(
            "RateDate",
            to_date(
                substring(
                    col("time_last_update_utc"),
                    6,
                    11
                ),
                "dd MMM yyyy"
            )
        )
        .drop("time_last_update_utc")
    )

    return df_normalized

In [0]:
def scd1_load(df_silver, table_name):
    cfg = METADATA_CONFIG[table_name]
    silver_path = f"{base_path}{cfg['silver_path']}"
    pk = cfg["primary_key"]
    try:
        existing_df = (
            spark.read.format("delta")
            .options(**storage_options)
            .load(silver_path)
        )
        display(existing_df)
    except Exception:
        print(f"[INFO] No existing Delta table for '{table_name}'. Running initial load...")
        df_silver.write.format("delta").mode("overwrite").options(**storage_options).save(silver_path)
        print(f"[INFO] Initial SCD1 Load Completed : {table_name}")
        return
    merged_df = (existing_df.alias("t")
                 .join(df_silver.alias("s"), pk, "leftanti")
                 .unionByName(df_silver)
    )
    merged_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true")\
        .options(**storage_options) \
        .save(silver_path)
    print(f"[INFO] SCD1 Merge Completed : {table_name}")

In [0]:
from pyspark.sql.functions import *
import functools

def scd2_load(df_incremental, table_name):
    cfg = METADATA_CONFIG[table_name]
    silver_path = f"{base_path}{cfg['silver_path']}"
    pk = cfg["primary_key"]     
    tracked_columns = ["FirstName","LastName","Email","Phone","City","State"]
    try:
        target_df = spark.read.format("delta").options(**storage_options).load(silver_path)
        display(target_df)
    except Exception:
        print(f"[INFO] No existing Delta table for '{table_name}'. Running initial load...")
        initial_df = (df_incremental.withColumn("start_date", current_date())
            .withColumn("end_date", lit(None).cast("date"))
            .withColumn("is_current", lit("Y"))
        )

        initial_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
            .options(**storage_options) \
            .save(silver_path)

        print(f"[INFO] Initial load complete for '{table_name}'.")
        return

    current_df = target_df.filter(col("is_current") == "Y")
    join_condition = current_df[pk] == df_incremental[pk]
    change_condition = functools.reduce(
        lambda a, b: a | b,
        [
            coalesce(current_df[c], lit("")) != coalesce(df_incremental[c], lit(""))
            for c in tracked_columns
        ]
    )
    changed_records = (current_df.alias("t").join(df_incremental.alias("s"),
            col(f"t.{pk}") == col(f"s.{pk}")
        )
        .filter(change_condition)
    )
    changed_keys = (
        changed_records
        .select(col(f"t.{pk}").alias(pk))
        .distinct()
    )
    expired_records = (
        target_df.alias("t")
        .join(
            changed_keys.alias("c"),
            pk
        )
        .filter(col("is_current") == "Y")
        .withColumn("is_current", lit("N"))
        .withColumn("end_date", current_date())
    )
    unchanged_records = (
        target_df.alias("t")
        .join(
            changed_keys.alias("c"),
            pk,
            "leftanti"
        )
    )
    new_versions = (
        df_incremental.alias("s")
        .join(
            changed_keys.alias("c"),
            pk
        )
        .withColumn("start_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit("Y"))
    )

    new_customers = (
        df_incremental.alias("s")
        .join(
            current_df.alias("t"),
            pk,
            "leftanti"
        )
        .withColumn("start_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit("Y"))
    )

    final_df = (
        unchanged_records
        .unionByName(expired_records)
        .unionByName(new_versions)
        .unionByName(new_customers)
    )

    final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true")\
        .options(**storage_options) \
        .save(silver_path)

    print(f"SCD2 Completed : {table_name}")

In [0]:
from pyspark.sql.functions import *

def schema_evaluation(df, table_name):
    cfg = METADATA_CONFIG[table_name]
    expected_columns = [c["name"] for c in cfg["columns"]]
    actual_columns = df.columns
    missing_columns = [
        c for c in expected_columns
        if c not in actual_columns
    ]
    extra_columns = [
        c for c in actual_columns
        if c not in expected_columns
    ]
    if missing_columns:
        return {
            "status": False,
            "missing_columns": missing_columns,
            "extra_columns": extra_columns,
            "df": df
        }
    return {
        "status": True,
        "missing_columns": [],
        "extra_columns": extra_columns,
        "df": df
    }

In [0]:
def process_bronze_to_silver(table_name):
    cfg = METADATA_CONFIG[table_name]
    bronze_path = f"{base_path}{cfg['bronze_path']}"
    silver_path = f"{base_path}{cfg['silver_path']}"
    reject_path = f"{base_path}rejects/{table_name}"
    watermark = get_watermark(table_name)

    df_bronze = (spark.read.format("delta")
        .options(**storage_options)
        .load(bronze_path)
    )
    if watermark:
        df_bronze = (
            df_bronze
            .filter(
                col("_IngestionTimestamp") > watermark
            )
        )
    records_read = df_bronze.count()
    latest_watermark = (df_bronze
                        .agg(
                            max("_IngestionTimestamp")
                            ).first()[0]
    )
    
    if df_bronze.count() == 0:
        print(f"No New Records : {table_name}")
        return

    if table_name == "exchange_rates":
        df_bronze = flatten_complete(df_bronze)
        df_silver = normalize_exchange_rates(df_bronze)
    else:
        df_silver = apply_type_cast(df_bronze,table_name)

    schema_result = schema_evaluation(
            df_silver,
            table_name
    )

    if not schema_result["status"]:
        print(f"Missing Columns : {schema_result['missing_columns']}")
        write_audit_log(table_name,records_read,0,records_read,"FAILED")
        return

    df_silver = schema_result["df"]
    df_silver = df_silver.withColumn("_ProcessedTimestamp",current_timestamp())
    
    df_silver, df_reject_pk = validate_records(df_silver,table_name)
    df_silver, df_reject_fk = validate_foreign_keys(df_silver,table_name)
    df_reject = df_reject_pk.unionByName(
        df_reject_fk,
        allowMissingColumns=True
    )

    records_valid = df_silver.count()
    records_rejected = df_reject.count()
    print(f"reject_df count: {records_rejected}")

    scd_type = cfg["scd_type"]
    if scd_type == "append_only":
        df_silver.write.format("delta").mode("append").options(**storage_options).save(silver_path)
    elif scd_type == "type1":
        scd1_load(df_silver,table_name)
    elif scd_type == "type2":
        scd2_load(df_silver,table_name)  
    
    if df_reject.count() > 0:
        df_reject.write.format("delta").mode("append").option("mergeSchema", "true").options(**storage_options).save(reject_path)

    write_audit_log(table_name,records_read,records_valid,records_rejected,"SUCCESS")
    update_watermark(table_name,latest_watermark)
    print(f"SUCCESS : {table_name}")

In [0]:
process_bronze_to_silver("customers")


reject_df count: 828
type2 scd
[INFO] No existing Delta table for 'customers'. Running initial load...
[INFO] Initial load complete for 'customers'.
SUCCESS : customers


In [0]:
process_bronze_to_silver("products")

reject_df count: 596
[INFO] No existing Delta table for 'products'. Running initial load...
[INFO] Initial SCD1 Load Completed : products
SUCCESS : products


In [0]:
process_bronze_to_silver("orders")

[INFO] Skipping duplicate PK check for append_only table 'orders'
reject_df count: 1157
SUCCESS : orders


In [0]:
process_bronze_to_silver("exchange_rates")